<br><br>

## 🟢 타이타닉.csv 데이터 파일로 실습하기  

In [2]:
import matplotlib.pyplot as plt 
import numpy as np 
import pandas as pd 

from sklearn.model_selection import train_test_split 
from sklearn.tree import DecisionTreeClassifier #중요도파악 
from sklearn.preprocessing import StandardScaler 

import os #파일이나 폴더경로를 정확하기 지정하려고 
train = pd.read_csv("titanic/train.csv")
test = pd.read_csv("titanic/test.csv")
print(train.shape)

#1.불필요한 열 삭제 , Cabin 은 결측치가 너무 많아서 제거하자 
print(train.head()) #원본데이터 inplace =True:안먹히는 함수가 많다.  
train = train.drop(columns=['PassengerId', 'Name','SibSp','Parch', 'Cabin'])
test  = test.drop(columns=['PassengerId', 'Name','SibSp','Parch', 'Cabin'])
print(train.head())
print(train.shape) #특성4개 삭제함 

#2. 결측치를 제거하자 
#1)결측치 확인하기 
print(train.isna().sum()) #각 특성별로 NaN개수가 출력된다. 
#Age - 177, Cabin-687, Embarked -2   
#Age나 Embarked 는 대체를
print(train.info())
print(train.describe()) #평균값이 나을지 중간값이 나을지를 지정하기 위해서 

#평균값으로 대체함 
age_mean = train["Age"].mean() 
train['Age'].fillna(age_mean, inplace=True) #자기자신이 바뀐다. 
test['Age'].fillna(age_mean, inplace=True)
print(train['Age'].isna().sum())
print(train.isna().sum()) 
print(test.isna().sum()) 
#Embarked는 행을 삭제시키자 

train = train.dropna(axis=0, how='any') 
#행중에 한 컬럼이라도 NaN값이 있으면 전체행을 삭제시켜라 
test = test.dropna(axis=0, how='any')

#2.이상치 제거 
#boxplot을 그려보자 
# train.boxplot() #데이터프레임이 내부적으로 몇개의 차트는 가지고 있다  
# plt.show() #이상치를 확인하기 위해 boxplot를 그려보자 

import numpy as np
def outfliers_iqr(data):
    q1,q3 = np.percentile(data,[25,75]) #percentile은 값 2개를 넘겨받을수있다
    iqr = q3-q1
    lower_bound = q1 -(iqr*1.5)
    upper_bound = q3 +(iqr*1.5)

    return lower_bound,upper_bound # tuple형태로 두값을 반환

#두개의 필드 Fare, Age필드가 이상치가 발견됨 
for i in ['Fare', 'Age']:
    lower, upper = outfliers_iqr(train[i])
    train[i][train[i]<lower] = lower 
    train[i][train[i]>upper] = upper 

for i in ['Fare', 'Age']:
    lower, upper = outfliers_iqr(test[i])
    test[i][test[i]<lower] = lower 
    test[i][test[i]>upper] = upper 

# train.boxplot() #데이터프레임이 내부적으로 몇개의 차트는 가지고 있다  
# plt.show() #이상치를 확인하기 위해 boxplot를 그려보자 


#3.원핫인코딩 
train = pd.get_dummies(train)
print(train.head())
print(train.columns)

#산포도행렬이든지 아니면 상관계수라도 
print(train.corr()) #- 상관계수를 구할 수 없는 필드들이 있어서 출력안됨 
# import seaborn as sns  #특성의 개수가 너무 많아서 메모리 부족임 
# sns.pairplot( train, diag_kind='kde', 
#               hue='Survived', palette='bright') 
# plt.show() 

#Survived가 젤 처음에 있음 
X = train.iloc[:, 1:] 
y = train.iloc[:, 0]
print(X.shape)
print(y.shape)

from sklearn.ensemble import RandomForestClassifier
model = RandomForestClassifier(n_estimators=100) 
model.fit(X, y)
print(model.score(X, y))


(891, 12)
   PassengerId  Survived  Pclass  \
0            1         0       3   
1            2         1       1   
2            3         1       3   
3            4         1       1   
4            5         0       3   

                                                Name     Sex   Age  SibSp  \
0                            Braund, Mr. Owen Harris    male  22.0      1   
1  Cumings, Mrs. John Bradley (Florence Briggs Th...  female  38.0      1   
2                             Heikkinen, Miss. Laina  female  26.0      0   
3       Futrelle, Mrs. Jacques Heath (Lily May Peel)  female  35.0      1   
4                           Allen, Mr. William Henry    male  35.0      0   

   Parch            Ticket     Fare Cabin Embarked  
0      0         A/5 21171   7.2500   NaN        S  
1      0          PC 17599  71.2833   C85        C  
2      0  STON/O2. 3101282   7.9250   NaN        S  
3      0            113803  53.1000  C123        S  
4      0            373450   8.0500   NaN    

/var/folders/gr/gm3dwjns4f35tjx2nc1z68m40000gn/T/ipykernel_20384/2397000970.py:31: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  train['Age'].fillna(age_mean, inplace=True) #자기자신이 바뀐다.
/var/folders/gr/gm3dwjns4f35tjx2nc1z68m40000gn/T/ipykernel_20384/2397000970.py:32: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting va

                  Survived    Pclass       Age      Fare  Sex_female  \
Survived          1.000000 -0.335549 -0.070165  0.313658    0.541585   
Pclass           -0.335549  1.000000 -0.327830 -0.713413   -0.127741   
Age              -0.070165 -0.327830  1.000000  0.132079   -0.082572   
Fare              0.313658 -0.713413  0.132079  1.000000    0.230325   
Sex_female        0.541585 -0.127741 -0.082572  0.230325    1.000000   
...                    ...       ...       ...       ...         ...   
Ticket_W/C 14208 -0.026409 -0.012534  0.001843 -0.022131   -0.024676   
Ticket_WE/P 5735  0.011485 -0.074656  0.062774  0.097048    0.014829   
Embarked_C        0.169966 -0.245733  0.034529  0.267138    0.084520   
Embarked_Q        0.004536  0.220558 -0.015230 -0.170732    0.075217   
Embarked_S       -0.151777  0.076466 -0.020667 -0.126586   -0.121405   

                  Sex_male  Ticket_110152  Ticket_110413  Ticket_110465  \
Survived         -0.541585       0.073942       0.034030    